# **Dataset Creation & Preprocessing**
### Ciaran O' Toole - C00297672
##### A portfolio produced as part of the Data Science & Machine Learning 1 Module in SE603, Bachelor of Science (Honours) Software Development at SETU
This notebook aims to outline my process of creating, refining and preparing my own custom movie dataset for use in this data science and machine learning portfolio. In order for me to gather the appropriate movie data, including titles, release dates, genres, production budgets, and revenues, the dataset will be constructed utilising data acquired from the TMDB and IMDb APIs.

The notebook will outline the several steps that go into creating a reliable and practical dataset, starting with the initial gathering and extraction of movie data and continuing with data cleaning to find and fix any missing, inadequate, or inappropriate records. To make sure the data is suitably organised and prepared for use in this portfolio, additional preprocessing will be done after the dataset has been revised.

In order to enable different machine learning models to be trained, the acquired data will be assessed against previously unseen records via the dataset being split into different training and testing subsets.

In addition to the technical aspects of the project, I intend to develop a dedicated website to showcase the findings. This website will act as the main platform for showcasing the dataset, showcasing the many models created and contrasting their effectiveness, and publishing the research findings.

### **Step 1 - The Initial Movie Data Collection**

This step establishes a connection to the TMDB API using authentication credentials stored separately in a configuration file to keep the API key secure. Here i then send multiple GET requests to the /movie/popular endpoint, we want to retrieve 1200 results so we can have 1,1,000 entries and 200 test entries which we will pick at random. Each page holds 20 results so we must cycle from page 1 -> 60 (60x20=1200). The data I retrieve is saved into a temporary list in which we will refine next. Note here we are grabbing everything, once we refine we will pick what we want to save so we dont have unnecessary data in our dataset.

In [29]:
from config import KEY,URL_START # always hide API KEY
import requests
import csv
import pandas as pand
import time

#set up
url = f"{URL_START}/movie/popular"
header = {"Authorization": f"Bearer {KEY}","accept":"application/json"}
movieselection = []

# obtain the default data for 1200 movies
# (page 1-60 as TMDB stores entries in pages
# of 20 entries as 60x20=1200)
for i in range(1, 61):
    response = requests.get(url, headers=header, params={"page": i})
    if response.status_code == 200:
        data = response.json()
        movieselection.extend(data["results"])
    else:
        print("connection failed.")
        print(response.text)



### **Step 2 - Refining The Collected Movie Data**

Now that the initial collection of 1,200 movies has been collected, the next step is to refine the information so it only contains the attributes relevant to the research question at hand. I iterate through each movie stored in my temporary movieselection list, using its unique TMDB ID to make an individual GET request to the movie details endpoint. This allows me to retrieve more detailed information about each movie than what was returned in the initial popular movies request.

For each successfully retrieved movie, I extract and store five attributes which I deem useful:

**Title:** The movie's name.

**Release Date:** When the movie was originally released to the public.

**Genres:** The genres associated with the movie, I store them all into a single string for cleanliness sake.

**Budget:** The production budget.

**Revenue:** The reported worldwide box office revenue.

These chosen fields are then appended to a new list called refinedmovieselection, creating a more relevant collection of information. If this segment of code fails to find a movie's data, it will report its ID and Title. I then print the total number of successfully retrieved movie entries. This allows me to check how many movies were retrieved.

In [31]:
# time to refine our movie data
refinedmovieselection = []

for entry in movieselection:
    url = f"{URL_START}/movie/{entry['id']}"
    refinedresponse = requests.get(url, headers=header)
    if refinedresponse.status_code == 200:
        refineddata = refinedresponse.json()
        # collect what we need to make our first questions prediction
        refinedmovieselection.append({
            "imdbID": refineddata["imdb_id"],
            "title": refineddata["title"],
            "release_date": refineddata["release_date"],
            "genres": ", ".join(genre["name"] for genre in refineddata["genres"]),
            "budget": refineddata["budget"],
            "revenue": refineddata["revenue"],
        })
    else:
        # incase we fail to pull a movie's data
        print("could not retrieve the data for:", entry["id"], entry["title"])
    time.sleep(0.30)

# helps view the total movie count retrieved
print("count:", len(refinedmovieselection))

count: 1200


### **Step 3 - Save My Custom Data Set to a CSV File**

With the movie information now refined and stored in refinedmovieselection, the next step is to store the collected data into a CSV file. I use Python's csv module to create a file named movies.csv. Before the data is written to the csv, I define the five column headings that will make up the dataset: imdbID, title, release_date, genres, budget and revenue. The csv file is opened using UTF-8 encoding, this allows me to use newline="" to prevent unwanted blank entries between records.

The data from the list is written into the csv using csv.DictWriter  The column headers are first created by writeheader(), and then each movie's dictionary is written as a distinct row by writerows(). I now have the first half of my data set complete.



In [32]:
# write to the csv file
with open("movies.csv", "w", newline="", encoding="utf-8") as file:
    fieldnames = ["imdbID", "title", "release_date", "genres", "budget", "revenue"]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(refinedmovieselection)

### **STEP 4 - IMDB Ratings Collection & Dataset Merge**
With the initial dataset created using the TMDb API, the next step is to merge it with additional information from IMDb's API, specifically movie ratings and their vote counts.

To do this I start by loading the existing movies.csv file using Pandas and extract every movie's title and IMDb ID into separate lists. I also print the total number of titles and IDs retrieved to verify that the expected 1,200 entries are here. Next, I load IMDb's title.ratings.tsv dataset, this allows me to link the two based on the tconst aka the IMDb ID.

Using Pandas' merge() function, I combine the two datasets by matching the imdbID. This allows me to retrieve the corresponding ratings and vote counts for each movie without having to manually match titles. A left merge is used here to make sure that all the movies from my original csv dataset are kept, even if a corresponding IMDb ratings entry cannot be found, we can fix this later in the cleansing stage.

I then replace my old movies csv dataset with the newly combined dataset and export it back into movies.csv.

In [33]:
# obtain every movie title from the csv
movies = pand.read_csv("movies.csv")
# obtain every movie title
movietitles = movies["title"].tolist()
movieIMDbIDs = movies["imdbID"].tolist()
# ensure every title and ID was retrieved (Should be 1200)
print("Total titles:", len(movietitles))
print("Total IDs:", len(movieIMDbIDs))

# now we read the tsv file of the IMDb ratings for movies
imdbratings = pand.read_csv("IMDb Ratings/title.ratings.tsv",sep="\t")
display(imdbratings.head())

combinedlistTMDbIMDb = movies.merge( imdbratings, left_on="imdbID",right_on="tconst", how="left")
# remove duplicate IMDbID column
combinedlistTMDbIMDb = combinedlistTMDbIMDb.drop(columns=["tconst"])
# quick overview of if it linked
display(combinedlistTMDbIMDb.head())
#set this new list as our new csv dataset
movies = combinedlistTMDbIMDb
movies.to_csv("movies.csv", index=False)


Total titles: 1200
Total IDs: 1200


,tconst,averageRating,numVotes
0,tt0000001,5.7,2242
1,tt0000002,5.4,330
2,tt0000003,6.4,2415
3,tt0000004,5.0,204
4,tt0000005,6.2,3117


,imdbID,title,release_date,genres,budget,revenue,averageRating,numVotes
0,tt33764258,The Odyssey,2026-07-15,"Adventure, Action, Fantasy",250000000,1804589435,8.4,579974.0
1,tt7526136,Heart of the Beast,2026-09-19,"Adventure, Thriller",82000000,192931039,7.1,61812.0
2,tt36586020,Other Mommy,2026-10-07,Horror,18000000,119807396,5.3,25373.0
3,tt35538033,Resident Evil,2026-09-16,"Horror, Science Fiction, Adventure",75000000,416368178,7.4,172459.0
4,tt22084616,Spider-Man: Brand New Day,2026-07-29,"Science Fiction, Action, Adventure",225000000,2523147515,7.9,390560.0
